In [31]:
import numpy as np
from iodata import load_one
from gbasis.wrappers import from_iodata
from gbasis.integrals.overlap import overlap_integral
from gbasis.evals.density import *
from grid import GaussChebyshev, BeckeRTransform, MolGrid, BeckeWeights

In [13]:
mol = load_one("../examples/peptide.fchk")

In [14]:
# 1-electron reduced density matrix (1-RDM) from SCF calculation
rdm = mol.one_rdms.get("scf")


In [15]:
coeffs = mol.mo.coeffs
occs = mol.mo.occs
rdm_raw = np.dot(coeffs * occs, coeffs.T)

In [16]:
mo = mol.mo
# Spin-resolved density matrices
# Alpha
rdm_a = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T

# Beta
rdm_b = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

# Total density
rdm_total = rdm_a + rdm_b

# Verify consistency
rdm_raw = (mo.coeffs * mo.occs) @ mo.coeffs.conj().T

assert np.allclose(rdm_total, rdm_raw)
assert np.allclose(rdm_total, rdm)

In [17]:
np.allclose(rdm_raw, rdm)

True

In [18]:
# basis function from mol object
basis = from_iodata(mol)

In [19]:
# basis function from mol object
basis = from_iodata(mol)

# AO overlap matrix
S = overlap_integral(basis)

# Number of electrons considering non-orthogonal basis
n_alpha = np.trace(rdm_a @ S)
n_beta = np.trace(rdm_b @ S)

print("Alpha electrons:", n_alpha)
print("Beta electrons:", n_beta)

# Check AO orthonormality
print(np.allclose(mo.coeffsa.T @ S @ mo.coeffsa,
                  np.eye(mo.coeffsa.shape[1])))

Alpha electrons: 19.99999998368834
Beta electrons: 19.99999998364403
False


In [ ]:
C = mo.coeffsa
occupied = mo.occsa > 1e-8

C_occ = C[:, occupied]
C_vir = C[:, ~occupied]

S_oo = C_occ.T @ S @ C_occ
S_vv = C_vir.T @ S @ C_vir
S_ov = C_occ.T @ S @ C_vir

print("Occupied-occupied:",
      np.max(np.abs(S_oo - np.eye(S_oo.shape[0]))))

print("Virtual-virtual:",
      np.max(np.abs(S_vv - np.eye(S_vv.shape[0]))))

print("Occupied-virtual:",
      np.max(np.abs(S_ov)))


Occupied-occupied: 8.716500963043844e-09
Virtual-virtual: 6.699490769567553e-06
Occupied-virtual: 9.97960156933377e-07


In [21]:
# Construct molecular grid
oned = GaussChebyshev(npoints=100)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_preset(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

# Evaluate spin-resolved densities
dens_a = evaluate_density(rdm_a, basis, grid.points)
dens_b = evaluate_density(rdm_b, basis, grid.points)

# Total electron density
moldens_raw = dens_a + dens_b
moldens = evaluate_density(rdm, basis, grid.points)

# Spin density
spin_dens = dens_a - dens_b

/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/atomgrid.py:905: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)


In [22]:
print("Alpha electrons:", grid.integrate(dens_a))
print("Beta electrons:", grid.integrate(dens_b))
print("Total electrons:", grid.integrate(moldens))
print("Total electrons (raw):", grid.integrate(moldens_raw))
print("Spin polarization:", grid.integrate(spin_dens))

Alpha electrons: 20.0001522877686
Beta electrons: 20.000152287724305
Total electrons: 40.00030458494785
Total electrons (raw): 40.000304575492905
Spin polarization: 4.432241955013424e-11


In [23]:
print("MO type:", mo.kind)
print("Maximum spin density:", np.max(spin_dens))
print("Minimum spin density:", np.min(spin_dens))
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens)))

MO type: unrestricted
Maximum spin density: 2.268368826108258e-11
Minimum spin density: -4.857270141656045e-11
Integrated absolute spin density: 6.542090336485095e-10


In [24]:
spin_dens_true = mol.one_rdms.get("scf_spin")
spin_dens_true = evaluate_density(spin_dens_true, basis, grid.points)
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens_true)))

Integrated absolute spin density: 4.200952171451098e-13


### Taylor Expansion Approximation of Exchange/Fermi Hole:

Apply the variable transformation $\textbf{r} = \textbf{r}_1$ and $\textbf{s} = \textbf{r}_2 - \textbf{r}_1$. The approxmiation of the Fermi hole is a spherical average over $\textbf{s}$ about the reference point $\textbf{r}$

$$
h_{X\sigma}(\textbf{r}, \textbf{r+s}) = - \frac{|\rho_\sigma(\textbf{r}, \textbf{r+s})|^2}{\rho_\sigma(\textbf{r})} \approx - \rho_\sigma(\textbf{r}) - \frac{1}{6} \left[ \nabla^2 \rho_\sigma(\textbf{r}) - 2\tau_\sigma + \frac{1}{2} \frac{(\nabla\rho_\sigma(\textbf{r}))^2}{\rho_\sigma(\textbf{r})} \right] s^2.
$$

Because the exchange energy only depends on the spherical average of the Fermi hole, the approximation of the Fermi hole is restricted to spherically symmetric models.

In [67]:
def exchange_hole(rdm, basis, grid, s, eps=1e-12):

    dens = evaluate_density(rdm, basis, grid.points)
    grad = evaluate_density_gradient(rdm, basis, grid.points)
    lap = evaluate_density_laplacian(rdm, basis, grid.points)
    tau = evaluate_posdef_kinetic_energy_density(rdm, basis, grid.points)

    print("Density:", dens.shape)
    print("Gradient:", grad.shape)
    print("Laplacian:", lap.shape)
    print("Kinetic energy density:", tau.shape)

    squared_grad = np.sum(grad ** 2, axis=1)
    clipped_dens = np.clip(dens, eps, None)

    hole = - dens + (-1/6) * (lap - 2*tau + (1/2)*((squared_grad) / clipped_dens))*s**2
    print("Hole:", hole.shape)
    return hole

The probability of finding another same-spin electron at the same position as the reference electron is completely extingushed
$$
h_{X\sigma}(\textbf{r}, \textbf{r}) = - \rho_\sigma(\textbf{r})
$$

In [ ]:
hole_a = grid.integrate(exchange_hole(rdm_a, basis, grid, s=0))
hole_b = grid.integrate(exchange_hole(rdm_b, basis, grid, s=0))

print("Integrated exchange hole (alpha):", hole_a)
print("Integrated exchange hole (beta):", hole_b)

electrons_a = grid.integrate(dens_a)
electrons_b = grid.integrate(dens_b)

print("Integrated electron density (alpha):", electrons_a)
print("Integrated electron density (beta):", electrons_b)

In [65]:
print(hole_a + electrons_a)
print(hole_b + electrons_b)

0.0
0.0


### Test Difference between Restricted and Unrestricted Functionals

In [26]:
res_mol = load_one("../examples/0001_q000_m01_k00_force_b3lyp_def2tzvpd.fchk")
unr_mol = load_one("../examples/0001_q000_m01_k00_force_ub3lyp_def2tzvpd.fchk")

res_basis = from_iodata(res_mol)
unr_basis = from_iodata(unr_mol)

In [27]:
res_mo = res_mol.mo
unr_mo = unr_mol.mo

In [28]:
res_rdm_a = (res_mol.mo.coeffsa * res_mol.mo.occsa) @ res_mol.mo.coeffsa.conj().T
res_rdm_b = (res_mol.mo.coeffsb * res_mol.mo.occsb) @ res_mol.mo.coeffsb.conj().T

unr_rdm_a = (unr_mol.mo.coeffsa * unr_mol.mo.occsa) @ unr_mol.mo.coeffsa.conj().T
unr_rdm_b = (unr_mol.mo.coeffsb * unr_mol.mo.occsb) @ unr_mol.mo.coeffsb.conj().T

In [29]:
oned = GaussChebyshev(npoints=100)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

res_grid = MolGrid.from_preset(
    atnums=res_mol.atnums,
    atcoords=res_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

unr_grid = MolGrid.from_preset(
    atnums=unr_mol.atnums,
    atcoords=unr_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

res_dens_a = evaluate_density(res_rdm_a, res_basis, res_grid.points)
res_dens_b = evaluate_density(res_rdm_b, res_basis, res_grid.points)

unr_dens_a = evaluate_density(unr_rdm_a, unr_basis, unr_grid.points)
unr_dens_b = evaluate_density(unr_rdm_b, unr_basis, unr_grid.points)

/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/atomgrid.py:905: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)
/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/atomgrid.py:905: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)


In [66]:
print(np.allclose(res_dens_a, unr_dens_a))
print(np.allclose(res_dens_b, unr_dens_b))

True
True
